# PlanTo3D — photoreal pass

Turns the geometrically-correct model into an architectural visualization,
using ControlNet to pin the generated image to the real massing.

**Set the runtime to GPU:** Runtime → Change runtime type → T4 GPU.

**What this stage is.** Everything before it measures: every wall, opening,
lawn and dimension traces back to the drawing. This stage *invents* — stone
coursing, dusk lighting, reflections, planting detail — because a floor plan
contains none of that. ControlNet keeps the invention pinned to our depth
and edges, so the result is this house rendered convincingly rather than a
plausible house that happens to look similar. Treat the output as an
impression of the design, not a measurement of it.

## 1. Setup

In [ ]:
import torch

print(
    f"GPU: {torch.cuda.get_device_name(0)}"
    if torch.cuda.is_available()
    else "NO GPU. Runtime -> Change runtime type -> T4 GPU, then rerun."
)

In [ ]:
!pip install -q diffusers transformers accelerate safetensors opencv-python-headless

In [ ]:
import sys
from pathlib import Path

repo = Path("/content/PlanTo3D")
if repo.exists():
    !cd {repo} && git pull --quiet
else:
    !git clone --quiet https://github.com/priyanshsoni096-blip/PlanTo3D.git {repo}

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

## 2. Upload the guides

Produced locally by `build_guides`, which writes `guide-render.png`,
`guide-depth.png` and `guide-edges.png`. Upload the depth map at minimum;
the shaded render is useful if you want to run image-to-image as well.

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
guides = {name: Image.open(name).convert("RGB") for name in uploaded}

depth = next((image for name, image in guides.items() if "depth" in name), None)
if depth is None:
    raise SystemExit("upload guide-depth.png")

# Diffusion works in multiples of 8; SD 1.5 is happiest near 512-768.
width, height = (d - d % 8 for d in depth.size)
depth = depth.resize((width, height))
print(f"guide size: {depth.size}")
depth

## 3. Load the model

Depth conditioning rather than edges: depth carries the massing and the
relative distance of every surface, so the model keeps our storey heights
and setbacks. Edge conditioning alone tends to preserve outlines while
flattening the form behind them.

In [ ]:
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()  # fits comfortably on a T4
print("pipeline ready")

## 4. Generate

`controlnet_conditioning_scale` is the dial that matters. High values hold
the geometry tightly but leave the image looking like a shaded model; low
values give richer materials and light while letting the building drift from
the plan. Around 0.8 keeps the massing recognisable with room for the
materials to breathe.

In [ ]:
from planto3d.photoreal import NEGATIVE_PROMPT, build_prompt

# Labels the pipeline actually read off the drawing, so the prompt describes
# this house rather than a generic one.
ROOM_LABELS = ["BEDROOM", "KITCHEN", "PARKING", "TERRACE GARDEN", "LANDSCAPE"]
STOREYS = 3

prompt = build_prompt(STOREYS, ROOM_LABELS)
print(prompt)

In [ ]:
generator = torch.Generator(device="cuda").manual_seed(7)

result = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=depth,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.8,
    generator=generator,
)

image = result.images[0]
image.save("photoreal.png")
image

## 5. Sweep the conditioning strength

Rather than guess the dial, render a few and pick. The trade-off is visible
immediately: fidelity to the plan against richness of the image.

In [ ]:
import matplotlib.pyplot as plt

strengths = [0.5, 0.7, 0.9, 1.1]
outputs = []

for strength in strengths:
    generated = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        image=depth,
        num_inference_steps=30,
        guidance_scale=8.0,
        controlnet_conditioning_scale=strength,
        generator=torch.Generator(device="cuda").manual_seed(7),
    ).images[0]
    generated.save(f"photoreal-{strength}.png")
    outputs.append(generated)

fig, axes = plt.subplots(1, len(strengths), figsize=(6 * len(strengths), 6))
for ax, strength, generated in zip(axes, strengths, outputs):
    ax.imshow(generated)
    ax.set_title(f"conditioning {strength}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Download

In [ ]:
for strength in strengths:
    files.download(f"photoreal-{strength}.png")